In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# LangChat Engineering Journal

> 每天一条，记录设计史。4周28天的认知演变。

---


## 2026-07-20（Week8-Day1：用户意图）

### 今天最大的认知
以前以为 LangChat 是"AI 运行时平台"，要负责编排所有系统。
现在知道 LangChat 是企业能力平台，被 Agent Host 直接调用，不做编排，不拥有业务数据授权。

### 今天最大的坑
OrchestratorAgent 曾经被写成"必经入口"，但它从未投入使用。ADR-001 §12 显式取代了这个表述。如果继续保留这个虚构中间层，所有 PRD 和 OpenSpec 都会基于不存在的链路展开。

### 今天最大的决策
Agent Host 和 LangChat 是"直接调用"关系。Orchestrator 是可替换角色，不是指定系统。

---


## 2026-07-21（Week8-Day2：ApplicationContract）

### 今天最大的认知
以前以为 ApplicationContract 就是接口定义——写清楚输入输出就够了。
现在知道 Contract 是业务治理一等对象：传输无关、版本不可变、治理语义（effect_policy / required_scopes / human_review_gate）嵌入其中。Contract 和 Version 两层分离是多版本共存的基础。

### 今天最大的坑
当前 SkillReleaseDescriptor 同时承担了三个角色（业务语义 + 设计描述 + 实现绑定），在 P0 阶段是合理简化，但随着能力增长会成为演进瓶颈。拆分的触发点应该是：当第一个 conditional_write 能力出现时，混合模型就无法只靠 frozen=True 保证安全了。

### 今天最大的决策
如果重新设计，会在 P0 阶段就引入 ContractVersion 概念，即使只有一个版本。因为后期从"没有版本"迁移到"有版本"的改造成本远大于一开始就加一层。

---


## 2026-07-22（Week8-Day3：Blueprint → Compiler → ExecutionPlanIR）

### 今天最大的认知
以前以为 Blueprint 是一份配置文件，Runtime 直接读取执行。
现在知道 Blueprint 是制品（artifact），有版本、digest、生命周期。Blueprint 和 ExecutionPlanIR 之间隔着 10 阶段确定性 Compiler——不是简单翻译。Compiler 的存在不是多余：它保证了同一输入永远产出同一输出。ExecutionPlanIR 是内部不可编辑的——不存在"手动 patch IR"的合法路径。

### 今天最大的坑
发现当前 10 阶段流水线大多为 pass-through stub（WP-03 阶段），每个阶段只是标记 "done" 并记录 Provenance entry，没有真实编译逻辑。这意味着代码验证了框架结构正确性，但实际编译能力需要后续 WP 填充。代码骨架和实际能力之间存在认知陷阱——看到 10 个阶段函数就以为它们在"做事"。

### 今天最大的决策
ExecutionPlanIR 的不可编辑性是整个确定性链条的支点。一旦允许"IR hotfix"，信任基础就崩塌了——无法审计、无法复现、无法回溯。如果重新设计，这条规则会被设为不可协商的第一原则。

---


## 2026-07-23（Week8-Day4：Runtime 无状态执行）

### 今天最大的认知
以前以为 Runtime 就是"跑代码的引擎"——加载代码、执行、返回结果，中间维护一些会话状态、用户上下文。
现在知道 Runtime 是"手术室"不是"病房"——无状态、封闭、所有信息通过 FrozenExecutionContext 带进来，所有结果通过 ExecutionResult 带出去。execute() 永不抛异常，所有失败都返回 fallback 七字段结构化结果。Runtime 包零 workflow import——执行框架通过参数注入，是可替换的插件。

### 今天最大的坑
发现 RuntimeLoader 是 WP-05 stub——它接受已实例化的 DeploymentRevision 直接返回，没有真实的 OCI pull、layer 验证、Compatibility Matrix Load check。这意味着当前是"被投喂"模式，调用者负责实例化。理解 Runtime 的无状态设计不难，但容易忽略 stub 和真实实现之间的巨大 Gap——看起来代码结构完整，但核心装载和验签能力尚未填充。

### 今天最大的决策
如果重新设计，无状态 Runtime + FrozenExecutionContext 不可变性 + 封闭性（零 workflow import）会被设为三大不可协商原则。无状态是水平扩展的前提，FEC 不可变是审计的基础，封闭性是可替换性的前提。三者缺一，平台就退化为"有状态单体应用"。

---


## 2026-07-24（Week8-Day5：Capability 与 Connector）

### 今天最大的认知
以前以为 Capability 就是 Plugin 的新名字——"即插即用的执行模块"。
现在知道 Capability 是**治理描述符**，不包含任何执行逻辑。E6 migration 后 `runtime_binding={}` 是铁证：Capability API 的 `/invoke`、`/invoke_stream`、`/executions/*` 全部移除，只剩 `/list_capabilities` 和 `/describe_capability` 两个元数据查询端点。真正的执行入口是 SkillRelease 的 canonical invoke。三层分离极其清晰：Capability（描述能做什么）→ SkillRelease（定义用它做什么）→ Workflow（内部实现）。

### 今天最大的坑
发现 MCP Connector 当前嵌在 Workflow 内部，没有独立治理。ADR-004 §8 描述的"Connector 只在 SkillRelease execution context 内可用"是目标态，代码现实是 Connector 作为 Workflow 的工具节点存在，没有独立的 effect_policy 校验、没有独立的版本管理、没有在 Platform Governance Plane 独立登记。`enforce_read_only()` 的 `_WRITE_INDICATORS` 是枚举式检测（`http_request`/`db_write`/`tool_call`/`provider_conditional_write`），无法覆盖所有可能的写操作形式。这是当前最大的架构债。

### 今天最大的决策
如果重新设计，Connector 独立治理应该更早做。当前 MCP 嵌在 Workflow 内意味着：Connector 调用不经独立 Capability Resolution、没有独立 effect_policy、版本管理依赖 Workflow。正确做法是 Connector 在 Platform Governance Plane 独立登记，每个 Connector 有自己的 effect_policy 和 scope，SkillRelease 通过 Capability Resolution 引用 Connector。E6 migration 证明了"Capability 不执行"是正确的——如果一开始就不给 Capability 执行能力，E6 迁移就不需要存在。

---


## 2026-07-26（Week8-Day7：Virtual CTO Review）

### 今天最大的认知
以前以为"ADR 通过了就等于实现了"。看到 ADR-001~004 的 G1-G18 全部通过，就以为 LangChat 的架构已经完整落地了。现在知道通过验证门的只是"当前态"（ADR-001~004），而"目标态"（ADR-005~008）全部还在评审中。v2 目标态的 6 个核心对象（BlueprintVersion、ExecutionPlanIR、SkillRelease v2、DeploymentRevision、ReleaseChannel、TrafficPolicy）在代码中完全不存在。ADR 的四态模型（文档事实/已确认方向/待决策/待验证）是一个极好的治理工具，它让"我们现在在哪"和"我们要去哪"同时存在而不矛盾。

### 今天最大的坑
发现五维评分中 Technical Debt 得分最低（6.5/10），根本原因不是某个单独的技术债项，而是系统性的：WorkflowSpec 作为"当前唯一执行格式但目标态要退役"的过渡期状态，意味着所有在 WorkflowSpec 上做的新功能都是"未来要迁移的债务"。这不是一个能快速修复的问题，需要通过 v2 制品链的逐步落地来系统性解决。

### 今天最大的决策
Week 8 建立了五维评分基线（综合 7.2/10），这个基线将在 Week 9-11 持续追踪。如果到 Week 11 综合分仍在 7.2 以下，说明 v2 制品链落地没有实质进展，需要升级为 P0 风险。

---


## 2026-07-25（Week8-Day6：完整链路图）

### 今天最大的认知
以前以为各模块是并列的组件图——Capability、SkillRelease、Runtime、Connector 各管各的。
现在知道它们是一条**不可缩短的串行链路**——10 个站点、7 个治理检查点、覆盖 6 个治理维度（身份认证/访问控制/数据安全/审计追踪/可靠性/性能保护）。每一步有独立的治理目的，去掉任何一步都打开具体的安全/审计/可靠性缺口。画完链路图后，Gap 无处藏身：核心执行链路（①-⑧）已对齐，但 Connector 治理（🔴）和 v2 制品链（🔴）是最大断裂点。

### 今天最大的坑
发现 `enforce_read_only()` 的递归扫描设计比想象中更重要——它不是简单的配置校验，而是运行时递归 8 层深度的安全扫描，检查键名和字符串值是否匹配 `_WRITE_INDICATORS`。这是 P0 阶段最后的安全防线。同时发现 Connector 治理是链路上最大的 Gap：MCP Connector 嵌在 Workflow 内部，没有独立的 effect_policy、scope 和版本管理，`_WRITE_INDICATORS` 枚举式检测无法覆盖所有写操作形式。

### 今天最大的决策
如果从零设计这条链路，会更早做三件事：① Connector 独立治理层（不嵌在 Workflow 内）；② v2 制品链从 P0 开始建（不走 WorkflowSpec 弯路）；③ ApplicationContract 在 P0 就引入（不让 SkillReleaseDescriptor 承担三个角色）。不会改变的设计：六维身份作为第一步、Read-Only 守卫作为最后防线、七字段结构化输出、幂等+限流在准备阶段。

---


## 2026-07-27（Week9-Day1：BlueprintVersion）

### 今天最大的认知
以前以为 BlueprintVersion 就是"有版本的 Blueprint"——加了个版本号而已。
现在知道 BlueprintVersion 的不可变性不是流程约定，而是**数学保证**：`@dataclass(frozen=True)` 在 Python 语言层面冻结 + SHA-256 内容寻址在密码学层面保证 + Registry `__post_init__` 自毁式防御在执行层面拦截。三层中任何一层被绕过，其他层仍然有效（防御性深度）。这是企业级代码的典范。

### 今天最大的坑
发现 ADR-005 D-2 定义的 Source Review（人工评审）在代码中**完全不存在**。当前只有 Admission（机器检查），从 Candidate 到 Version 的升级路径缺少人工评审门。这意味着理论上任何通过机器检查的 Candidate 都能自动升级为 Version——治理缺口。不过考虑到当前是 WP-02 阶段（Blueprint 基础设施），Source Review 可能在后续 WP 中补充。

### 今天最大的决策
如果重新设计，会保留 BlueprintVersion 的所有核心设计：frozen=True、SHA-256 内容寻址、前向唯一生命周期、Registry 无执行方法 + 自毁防御、评审两段式 + 不检查业务正确性。唯一可能调整的是：增加 Candidate withdraw 状态（允许作者主动撤回 In Review 的 Candidate），以及 Registry 用 event-sourced 持久化模式。


## 2026-07-28（Week9-Day2：SkillRelease 唯一可部署单元 + DDD 战略设计验证）

### 今天最大的认知
SkillRelease 的"唯一可部署单元"地位不是技术决定，而是治理决定。它把制品链上的所有治理（确定性构建、依赖锁、评估、审批、签名）汇聚到一个不可变制品上。同时，从 DDD 战略设计视角看，MI 的 17 个 Bounded Context 划分经得起 DDD 原则检验——Core Domain（P0）/ Supporting Domain（P1）/ Generic Domain 的分层已经隐含在 Domain Model 中。

### 今天最大的坑
WorkflowSpec binding（W01-W09）当前是 SkillRelease 底层执行态，但目标态要退役。当前代码事实和目标态设计之间存在"过渡期认知 gap"。DDD 战略设计验证中发现：17 个 Context 有数据流边界但缺少显式的 Context Map（跨 Context 集成契约语义不够清晰）。

### 今天最大的决策
理解了 SkillRelease 和 DigitalEmployeeDefinition 的分离原则：定义是"谁"，SkillRelease 是"什么"。这与 DDD 中 Aggregate 边界划分是同一个思维模式——DigitalEmployeeDefinition 是引用语义锚点而非巨型聚合根，正如 DDD 中 Aggregate 应尽可能小。


## 2026-07-29（Week9-Day3：Deployment / DeploymentRevision）

### 今天最大的认知
以前以为 Deployment 就是"把 Release 部署到服务器上"，是一个运维动作，不是架构对象。Release 和 Deployment 是"发布"这一个动作的前后两步。
现在知道 DeploymentRevision 是 Runtime 层最重要的架构对象——它是一个不可变的、内容寻址的完整执行闭包，16 个字段锁死了"这次执行用了什么、跑了什么、在什么环境下跑的"。它是 Supply Chain 和 Runtime 之间唯一的合法通道。SkillRelease 是通用制品（跨环境可移植），DeploymentRevision 是特定实例（绑定到具体环境）。合并它们意味着每次环境变更都要重新走 Supply Chain（构建、评估、签名），这是不可接受的。

### 今天最大的坑
发现回滚语义是"前向操作"而不是"还原操作"——从历史 DeploymentRevision digest 闭包物化新 Revision，不改变历史对象状态。这与传统 ERP 的"还原数据库"完全不同。一开始觉得这很绕，但理解后意识到这是审计完整性的数学保证：你不修改历史，只创造新的决策记录。另一个坑是 source_channel 只作 provenance，不进 runtime closure digest——这意味着 Channel 名变化不改变 Revision 身份，但它记录了"这个 Revision 是从哪个 Channel 晋升来的"审计线索。

### 今天最大的决策
如果重新设计，会把 DeploymentRevision 的 16 字段闭包作为不可协商的第一约束。AI 应用的执行结果不确定性远超传统软件——受 prompt、模型版本、知识库快照、策略叠加影响。如果 DeploymentRevision 不把这些全部锁死，就无法做到"同一个闭包 → 同一个执行结果"，灰度对比和回滚就不可靠。evaluation_only 默认值为 True 是一个安全设计典范——默认隔离，生产部署需要显式打开。


## 2026-07-30（Week9-Day4：ReleaseChannel / TrafficPolicy）

### 今天最大的认知
以前以为灰度发布是部署工具的功能——在 CI/CD 流水线里配个百分比就行了。现在知道 LangChat 把灰度拆成了两个独立架构对象：ReleaseChannel（Supply Chain 层的晋升指针）和 TrafficPolicy（Runtime 层的流量策略）。两者在架构上完全解耦：Channel 移动不改变流量，TrafficPolicy 不读 Channel。这个"标记正式版"≠"全量上线"的中间态，才是灰度发布的工程价值。传统 ERP 的"发布"把版本标记、部署、切流混在一个动作里，LangChat 拆成三个独立动作（晋升→物化→切流），各自独立审计。

### 今天最大的坑
发现 ReleaseChannel 被严格归在 Supply Chain 层，不在运行时请求路径中。一开始觉得奇怪——"正式版指针"难道不是运行时关心的吗？但理解后意识到：如果 Runtime 读 Channel，那 Channel 移动就会直接影响流量——这恰恰是设计要避免的。Channel 是"版本管理团队的事"，TrafficPolicy 是"运维团队的事"。代码验证了这一点：TrafficPolicy 在构造时就拒绝一切非精确引用（latest、channel 名、mutable name 全被 ValueError 拒绝）。

### 今天最大的决策
灰度的核心不是"能不能按比例切流量"，而是"版本标记与实际运行状态的解耦程度"。如果重新设计 MI 的合同审核机器人发布流程，会把"标记正式版"（ReleaseChannel）和"实际切流"（TrafficPolicy）分成两个审批流——版本管理委员会标记正式版，运维团队根据灰度策略逐步切流。出错时回滚只需新建 TrafficPolicy 版本指向旧 Revision，不需要动 Channel 指针。


## 2026-07-31（Week9-Day5：DigitalEmployeeDefinition）

### 今天最大的认知
以前以为数字员工就是一个"智能体"——定义它、启动它、它就开始干活。现在知道 DigitalEmployeeDefinition 只是"语义锚点"——一组引用的集合，指向真正干活的对象（SkillRelease → DeploymentRevision → Execution）。定义不拥有 Runtime，不持有 Deployment 状态，不构造 FrozenExecutionContext，不充当执行入口。当前代码的 `status=active` 同时承载了"定义已发布"和"部署在服役"双重语义——目标态要拆成 `DigitalEmployeeDefinition.Published` + `Deployment.Active` 两个独立状态。这和传统 ERP 里"员工主数据 ≠ 流程实例"是完全一致的道理。

### 今天最大的坑
当前代码的 `kill_switch` 直接放在 DigitalEmployeeModel 上，看似合理。但从目标态看，kill_switch 属于 Deployment 层——"停止运行"是运行时决策，不是定义层决策。定义可以 Deprecated（退役），不能被"停止运行"，因为定义本身不运行。另外 `bound_skill_id` 是 tag 引用而非 digest 引用，违反了"引用而非持有"原则——目标态要改成 digest/版本指针。

### 今天最大的决策
如果给 MI 的 10 个 mall 都部署"合同审核数字员工"，正确的做法是：1 个 DigitalEmployeeDefinition（共享身份声明）+ 10 个 Deployment（每个 mall 一个）。每个 Deployment 有独立的 KnowledgeSnapshot、PolicyBundle、DeploymentRevision。知识库更新一个 mall 不影响其他。这正是"定义不拥有 Runtime"的核心价值——多环境部署时定义保持稳定，运行独立演进。

---


## 2026-08-01（Week9-Day6：Domain Model Diagram 动手交付）

### 今天最大的认知
以前以为 Domain Model 就是 ER 图加类图——把数据表关系画清楚就够了。
现在知道 Domain Model 是**治理拓扑图**：不仅画数据关系，还要画生命周期（谁先出生谁先死）、不变量约束（合并后哪些不变量会破裂）、层归属（哪个对象在四层中的哪一层）、禁止职责（这个对象绝对不能做什么）。一张好的 Domain Model Diagram 能让你在 3 分钟内回答"这个对象放这里合不合适"。

### 今天最大的坑
代码验证发现 Runtime Layer 覆盖度只有 10%——Deployment、DeploymentRevision、TrafficPolicy、FrozenExecutionContext 四个核心对象在代码中完全不存在。当前代码的"数字员工"直接从 SkillRelease 跳到 Execution，中间没有部署闭包。这意味着所有 v2 治理（灰度、回滚、多环境、闭包 digest-pin）在当前代码中都没有基础设施支撑。Blueprint 层（SC-02/03）是代码最成熟的部分，但从 SkillRelease 之后几乎是断崖。

### 今天最大的决策
合并/拆分判定原则：**如果两个对象的演化节奏不同、生命周期不同、变更 Owner 不同，它们就不应该合并。** 用这个框架可以快速判断：Capability/CapabilityRelease 可短期合并（小团队），但 BlueprintVersion/ExecutionPlanIR 绝不可合并（违反 HC-3 单向制品链）。Definition/Deployment 绝不可合并（否则多环境部署变成不可能）。这条原则将作为后续架构评审的标尺。

---


## 2026-08-02（Week9-Day7：Virtual CTO Review — ADR Health Check）

### 今天最大的认知
以前以为"ADR 通过了就等于稳定了"——只要状态是 accepted，就可以放心引用。
现在知道 ADR 体系本身也有健康度问题：两套编号体系并存（品牌 ADR-00X vs 技术 ADR-LC-0XX vs v2 战略 ADR-00X）、v2-ADR-005/007/008 覆盖面过大需要拆分、v2-ADR-001~004 长期停留在"评审中"/"文档事实"状态阻碍下层推进。ADR Health Check 不只是看"有没有过时"，更要看"覆盖面是否合理"、"编号体系是否一致"、"实施状态是否可追踪"。

### 今天最大的坑
五维评分从 Week 8 的 7.2 降到 6.8。初看像是退步，实际上是"达克效应的正向穿越"——Week 8 看到的是链路全景（框架完整），Week 9 拆开每个对象后发现"框架完整但内容空洞"（Runtime Layer 覆盖度仅 10%）。评分下降不是架构变差了，是理解更深了。最大的坑是 v2-ADR-007（RuntimeABI + CompatMatrix + FEC wire）把三个独立技术决策合在一个 ADR 里，任一子主题修订都要整个 ADR 重评——这是治理设计的技术债。

### 今天最大的决策
提出三条 CTO 级建议：① 推进 v2-ADR-001~004 正式冻结（已通过 G1-G18 验证门，继续"评审中"不带来额外谨慎）；② 拆分 v2-ADR-007 为三个独立 ADR（RuntimeABI / CompatMatrix / FEC wire）；③ 在每个 ADR 中增加 implementation_status 字段（draft/partial/implemented/verified），让"ADR 定义了但代码不存在"这个最大风险可见化。

---


## 2026-08-03（Week10-Day1：Permission & Policy — 谁允许谁做什么）

### 今天最大的认知
以前以为权限就是 RBAC——定义角色、分配权限、代码里 if-hasPermission 检查。
现在知道在企业 AI 平台里，权限不是功能模块，而是**横切四层的治理制品链**：Policy（单条规则）→ PolicyBundle（不可变策略束，Pydantic frozen+strict）→ SkillRelease（打包）→ DeploymentRevision（digest-pin）→ FEC（冻结）→ Runtime 只读 → Dual-Gate 9步算法验证。三个维度同时生效：静态维度（角色权限映射表）、制品维度（conditional_write 必须配 review_gate 的跨字段不变量）、执行维度（Step 7 审批后用冻结快照重跑 Step 0-4）。传统 RBAC 的"检查"只是这条链的最后一环。

### 今天最大的坑
Dual-Gate 的 9步算法看起来极其复杂，但深入后发现它的核心洞察只有一个：**审批覆盖的是字节级快照**。Step 7 不只是"审批通过了就执行"，而是用冻结的 invocation_context_canonical_json 重跑 Step 0-4，对比 digest。这意味着如果审批者批准了 amount=5000，调用方在恢复时改成 amount=100（试图绕过 cannot max_amount=1000），系统会用冻结的 5000 做验证——篡改被拒绝。这个设计的复杂度是"AI 时代安全"的必要代价，不是过度设计。

### 今天最大的决策
Permission 不放 Runtime 里的根本原因不是"关注点分离"这种架构审美，而是**安全必要性**：LLM 可以通过 Prompt Injection 影响运行时行为，如果权限检查也在运行时，就被攻击面覆盖了。权限必须在外部冻结好，Runtime 只是一个无法篡改的执行者。这条原则对 MI 的启示是：合同审批数字员工的权限策略必须在部署前定义，不能"运行时根据情况灵活调整"——"灵活"在 AI 平台中等同于"可被攻击"。

---


## 2026-08-04（Week10-Day2：Audit & Trace）

### 今天最大的认知
以前以为 Trace 就是高级日志——把 `logger.info()` 换成 `span.record()`，本质没区别。
现在知道 Trace 和日志是两个物种。日志是叙述（面向人类阅读），Trace 是证据（面向系统查询）。日志没有结构，Trace 有严格的 Span Tree（trace_id / span_id / parent_span_id）。更关键的是：Trace 是 Governance 的基础设施——没有 Trace，Permission 效果无法验证、Approval 决策无法追溯、PII 泄漏无法定位。Trace 不是排障工具，是治理证据链。

### 今天最大的坑
LangChat 的 ExecutionSpan 有 10 种 SpanKind，覆盖所有执行路径（workflow_run / llm / rag_retrieval / channel_dispatch / capability_invoke / mcp_tool 等）。这不是日志级别（DEBUG/INFO/WARN），而是业务语义类型。坑在于：如果你用传统日志思维去理解 Trace，你会忽略 SpanKind 的语义价值——你不会想到可以 SQL 查询"所有租户 A 的失败 LLM 调用"，也不会想到可以按 kind 聚合分析"RAG 和 LLM 哪个更慢"。

### 今天最大的决策
LangChat 选择"OTel 形状 + 自实现"而不是直接用 OpenTelemetry SDK，这个决策的核心逻辑是：获得 OTel 的兼容形状（未来可以插入 OTel/Langfuse exporter），但不背负 OTel SDK 的重量级依赖。DbSpanEmitter 异步批量写入、CompositeEmitter 支持多写、contextvars 保证 async-safe 传播——这些都是自实现才能精确控制的。对 MI 的启示是：当数据量到 70 万 Span/天时，自实现的控制力（保留策略 CLI、租户隔离查询、预聚合）比依赖外部平台更重要。

---


## 2026-08-05（Week10-Day3：Approval — 人审）

### 今天最大的认知
以前以为人审就是一个审批按钮——AI 生成建议，人点"通过"，就这么简单。
现在知道 LangChat 的人审是一个**三层治理结构**：① 制品层（Release Gate 的 Approval attestation，digest-pin 到技能制品上，不审批不发布）；② 运行时层（HITL Gate，conditional_write 触发 `pending_human_review` 状态机强制暂停，6 步验证 + 4 写原子 CAS）；③ 部署层（DeploymentRevision 的 `ApprovedDeploymentRevision` 类型级边界）。三层各有各的强制机制：制品层用 Gate 序列 monotonic 约束、运行时层用六状态原子状态机、部署层用 frozen dataclass 类型系统。Approval 和 Signature 的分离是精妙设计——治理决定（人）和密码学证明（机器）分离验证。

### 今天最大的坑
`register_revision_from_envelope` 的设计让我踩了一个认知坑：它从 DB 行实际状态派生 approval，而不是从请求参数 `auto_approve` 派生。最初我以为 `auto_approve=True` 会自动批准——但它不会，因为 idempotent re-insert 返回已有的 pending 行，approval 从 DB 实际状态读取。这是防"重新注册时绕过审批"的安全设计。另一个坑：HITL 的 SameTransaction CAS 做了 4 个写操作（GateChallenge 锁 → Execution 锁 → token 消费 → event insert + CAS），全在一个事务里——两个并发审批会在 SELECT FOR UPDATE 上串行化，失败方收到 ChallengeAlreadyDecidedError。这比传统 ERP 的"审批后写日志"复杂得多，但这是 AI 时代的必要安全代价。

### 今天最大的决策
理解了"为什么 AI 不能全自动发布"的核心论点：不是不信任 AI 的能力，而是企业治理需要**可追溯的责任链**。传统 ERP 的审批可以被管理员配置跳过，但 LangChat 的审批由类型系统（ApprovedDeploymentRevision）+ 状态机（pending_human_review 不可跳过）+ Descriptor 验证器（conditional_write 强制 review_gate）三层强制执行。对 MI 的启示是：合同审批数字员工的技能发布、租金变更建议的执行，都必须经过人审——这不是降低效率，是建立企业信任边界。没有这个边界，AI 永远只能做 Demo，不能进生产。

---


## 2026-08-06（Week10-Day4：Fail-closed vs Fail-open + Approval）

### 今天最大的认知
以前以为 fail-closed 就是"出错就报错"，fail-open 就是"出错就跳过"，是个二元选择。
现在知道 LangChat 的 fail-closed 是**四层分层设计**：① 安全边界层（认证/授权/hash/scope）绝对 raise；② 制品治理层（审批/类型边界/状态机）结构性不可绕过；③ 执行层（LLM/KB/工作流）优雅降级返回兜底结果；④ 风险保留层（兜底结果中的敏感关键词检测）即使降级也不丢失风险信号。最精妙的设计是 `_fallback_result()`——`execute() MUST NEVER raise to the caller`，但这不是 fail-open，而是 API 契约：执行失败是结果不是异常，风险通过 `human_review_required` + `risk_flags` 保留。

### 今天最大的坑
`auto_approve_on_timeout=False` 看起来只是一个默认值，改 True 就行——但**默认值就是架构决策**。传统 ERP 审批超时自动转交或自动通过很常见；在 AI 平台里，这是不可接受的。AI 生成的建议可能看起来合理但实际有害，所以超时 = 保持 pending = 需要人工处理。另一个坑：`register_revision_from_envelope` 从 DB 行实际状态派生 approval，不信任请求参数 `auto_approve=True`——因为 idempotent re-insert 返回已有的 pending 行，approval 从 DB 实际读取。这是防"重新注册时绕过审批"的安全设计。

### 今天最大的决策
Fail-closed 在 MI 商业地产场景的具体映射：① 合同审核技能上线前必须法务审批（DeploymentRevision Approval Gate）；② 租金调整建议必须人工确认后才写入 ERP（HITL Gate + conditional_write）；③ 跨商场数据访问必须被拒绝（Capability scope 校验 → GatewayError）；④ LLM 挂了但输入含"退款/解约/减免"等敏感词 → 返回兜底结果但 `human_review_required=True`。MI 的架构原则应该是：**安全问题绝不妥协，执行问题优雅降级，但风险信息不丢失。**

---


## 2026-08-07（Week10-Day5：Realization Rollback + FrozenExecutionContext）

### 今天最大的认知
Realization Rollback 不是 DELETE，而是六种对象六种归档策略。Workflow → archived，Version → is_published=False，Binding → is_active=False，Assistant → archived，KB → 只解除关联不删数据，Prompt → 指针回退或模板退休。每一种策略都尊重对象的语义和历史可审计性。FrozenExecutionContext V2 不是一个简单的 immutable object——它是一个密码学容器，13 个 digest 把它跟 SkillRelease、DeploymentRevision、PolicyBundle 绑死，任何篡改都会导致 digest 不匹配。这不是代码层面的「君子协定」，而是数学层面的完整性保证。

### 今天最大的坑
`__wrapped__` 的使用。Realization Orchestrator 里 `create_workflow`、`create_version`、`publish_version` 都用了 `getattr(_create_workflow, "__wrapped__", _create_workflow)`——第一眼看像是绕过了什么安全机制。实际上是因为 `@with_session` 装饰器会自动 `db.commit()`，而在 savepoint 隔离模型中，任何中间 commit 都会打断事务完整性。所以必须用 `__wrapped__` 绕过装饰器的自动提交，保持所有 mutation 在同一个 savepoint 内。这是事务完整性的需要，不是安全绕过。另一个坑：Rollback 里 KB 的处理是「metadata flag」而非归档，因为 KB 里的文档可能已经被工作流使用过，硬归档会导致历史引用断裂。

### 今天最大的决策
MI CRE 场景中 Realization Rollback 的应用：当「租户续签数字员工」的 Prompt Template 引用了过期租金系数表时，Rollback 不是删除 v3 模板——而是指针回退到 v2，v3 行保留可查。修正后重新 Realize 生成 v4，attempt 递增。完整审计链零数据丢失。这比传统 ERP「删了重录」强太多。FrozenExecutionContext 对应到 MI 场景就是「合同审批身份快照」：审批人的权限范围、委托链、策略快照在审批过程中不可变——这直接对应企业内控对审批流程的合规要求。

---


## 2026-08-08（Week10-Day6：Governance 覆盖图 + Gap 分析）

### 今天最大的认知
以前以为 Governance 是一个模块，和 Knowledge Base、Workflow 并列。
现在知道 Governance 是三个时间轴上的横切约束：Build Time（Custody + Rollback + Plan）→ Deploy Time（PolicyBundle + Release Gate + Compat Matrix）→ Runtime（SixDim + Dual-Gate + ReadOnly + Audit + Trace + PII + Retention）。15 个治理检查点分布在 ADR-007 三段架构链的每一段。这不是"模块化治理"，是"空气化治理"——每一层都呼吸它，但你看不到它独立存在。

### 今天最大的坑
PII Redaction 默认关闭。代码写得很漂亮——`RedactionStrategy` Protocol → `NoopRedactionStrategy`（默认）→ `RegexRedactionStrategy`（8 种 PII 模式：EMAIL/IP/PHONE/ID_CARD/BANK_CARD/URL_CRED）→ entry point 插件扩展——但 `NoopRedactionStrategy` 是默认值！生产环境如果忘记开启 `PII_REDACTION_ENABLED`，用户敏感数据裸奔到 Trace Payload。这不是代码问题，是架构决策问题：定位为"企业 AI 应用平台"（ADR-001），PII 保护不应该是 opt-in。另外发现 Enterprise Systems 侧出站签名缺失——Tool Use 调用外部系统没有签名，企业系统侧无法验证来源，这在多租户场景下是安全隐患。

### 今天最大的决策
画出了完整的 Governance Coverage Map，识别了 7 个 Gap。最大的 Gap 优先级判定留给明天 Virtual CTO Review。初步判断：PII 默认关闭 > 出站签名缺失 > 合规报告缺失 > 数据分级保留 > 跨租户测试覆盖。核心原则：Governance 不是"有没有"的问题，是"默认开不开"的问题。安全机制默认关闭 = 没有安全机制。

---


## 2026-08-09（Week10-Day7：Virtual CTO Review — Governance 周总评）

### 今天最大的认知
以前以为 Governance Review 就是检查"有没有治理"——列出治理机制，确认存在就完了。
现在知道 Governance Review 要回答四个问题：① 治理覆盖了哪些层（Coverage Map）？② 每一层是默认开还是默认关（Default Policy）？③ 治理机制之间是否形成闭环（Custody→Evidence→Verify）？④ 如果只能修一个，先修哪个（Priority Matrix）？这四个问题的答案构成了一个完整的治理评估框架。本周的 Review 给出了明确答案：覆盖 8/15 个检查点、PII 默认关闭是最大风险、修复成本一行代码、优先级判定完成。

### 今天最大的坑
五维评分三周持平在 6.8（Week 9 = Week 10 = 6.8），初看像是"没有进步"。但深入分析发现：理解深度从 7.5 → 8.5（↑1.0），Charter 对齐度从 25% → 37.5%（+12.5%），新增 10 个认知增量。评分持平是因为"发现的好消息"（FEC 已落地、Dual-Gate 精密）和"发现的坏消息"（PII 默认关闭、出站签名缺失）相互抵消。这是达克效应正向穿越的第三阶段：不再盲目乐观也不盲目悲观，而是精确知道已有什么、缺什么。

### 今天最大的决策
PII Redaction 默认开启 — 优先级判定为 P0。判定依据四条：① 不对称风险（调试不便 << 数据泄漏+合规违规）；② 定位一致性（ADR-001 说"企业平台"，PII 不能 opt-in）；③ 修复成本极低（改一个默认值）；④ 行业先例（Azure/AWS/Google 全部默认开启）。修复行动方案四步已定：改默认值 → 启动日志 → 创建 ADR-LC-014 → 增加关闭审批流程。这不是"Week 11 的任务"，这是"今天的任务"。

---


## 2026-08-10（Week11-Day1：Capability Inventory — 能力清查）

### 今天最大的认知
以前以为 Capability Catalog 是 LangChat 的能力目录，列出了平台所有能力，SkillRelease 是能力的具体实现。
现在知道代码里有三套"能力注册体系"并行存在：① Capability Catalog（2 条，runtime_binding 全空，纯展示壳）；② SkillRelease Registry（10 条可执行 Skill，各有 executor_fn）；③ Capability Gateway（W01 MCP 独立体系）。三者互不引用，平行存在。Capability Catalog 的 2 个条目在 E6 迁移后变成了被掏空的元数据——OpenSpec 明确写 "runtime_binding SHALL be the empty object {}"。真正的执行入口是 SkillRelease 的 POST /v1/skill-releases/{skill_id}/invoke。Capability API 默认关闭（CAPABILITY_API_ENABLED = False），因为 Catalog 还没准备好面对外部。

### 今天最大的坑
W01-W09 的 Skill ID 看似遵守了 ADR-003 正交约束——没有行业词（ops.anomaly、customer.escalation、brand.research）。但 workflow_binding 里全是 mall-* 前缀，display_name 全是 "Mall Ops / Mall Customer / Mall Brand"。**Skill ID 遵守了正交约束的字母，但违反了精神。** 这些不是 Capability（跨行业原子能力），它们是 Application 级别的打包——9/10 个 Skill 是商业地产专属，只有 workflow.execute 一个是真正平台级的。ADR-003 说 Application = Capability 子集 × Industry 标签，但代码里这个矩阵没有被显式表达。

### 今天最大的决策
Week 11 的打开方式不是"继续学新东西"，而是"面对代码事实"。前三周建立了漂亮的心智模型——四层架构、25+ 个目标态对象、ADR 正交模型。今天开始把这些模型和代码对照，发现 Gap 不是"目标态对象没实现"（这在意料之中），而是"当前态有三套平行体系谁也不认谁"。Capability Catalog 不投影 SkillRelease，SkillRelease 不查 Catalog，Gateway 自成一套。这才是最危险的 Gap——不是缺什么，而是现有的东西彼此不连接。

---


## 2026-08-11（Week11-Day2：Gap Matrix — 目标态 vs 代码实现）

### 今天最大的认知
以前以为 Gap 分析是"列一个完成度百分比表"。
现在知道 Gap 分析的真正价值是识别**"语义 Gap"**——不是"有没有"的问题，而是"在那里但语义不对"的问题。最危险的 Gap 不是"对象不存在"，而是 DeploymentRevision 以为闭包完整实际不完整——没有 KnowledgeSnapshot 和 PolicyBundle 可引用，却按"digest-pinned 闭包"的语义运行。这给了一种虚假的安全感。整张 Matrix 打分下来，Supply Chain 是最薄弱的层（11个目标对象只有 2 个 ≥7分），而它恰恰是平台的"配置管理 + 变更管理"——ERP 里叫"工程变更管理"。

### 今天最大的坑
SkillRelease canonical execution 有 3710 行代码，初看以为这是 v2 制品链最厚的部分，应该没问题。但仔细看发现：那是 v1 canonical 执行路径（执行/审批/限流/重放/HITL 全套），**不是 v2 SkillRelease 作为 OCI 制品**。v2 的 OCI 打包（supply_chain/oci/）只有 manifest + publish 骨架（~400行），和 canonical execution 没有打通。同一个名字"SkillRelease"，在 v1 和 v2 指的是不同的东西。这种"术语重叠陷阱"比"完全空白"更危险——因为它让人误以为已经实现了。

### 今天最大的决策
如果给 3 个 Sprint 补 Gap，优先级排序：① 先修 KnowledgeSnapshot（部署闭包不完整 = 执行不可复现，最高运行时风险）；② 再修 RuntimeABI（Runtime 和制品之间无版本契约 = 升级必爆炸）；③ CapabilityRelease 可延后一个 Sprint（Capability 现在可用，版本化发布缺失的迁移成本可控）。判定依据：安全性 > 完整性 > 美观性。KnowledgeSnapshot 直接影响"上周三的数字员工回答了什么"——回答不了就是合规风险。


## 2026-08-13（Week11-Day4：Knowledge 现状）

### 今天最大的认知
以前以为 RAG 工程能力强 = Knowledge 治理没问题。现在知道这是两个完全不同的维度。当前 RAG 管道（Query Transform + Hybrid Search + Rerank + Citation + Evaluation + KB Improvement Queue）工程做得确实很好，但治理层完全空白：没有 KnowledgeCollection（逻辑集合）、没有 KnowledgeSnapshot（不可变快照）、没有 digest、没有被 DeploymentRevision 闭包锁定。知识库随时可改，数字员工行为不可预测——这在企业场景是合规风险。

### 今天最大的坑
当前 KnowledgeBaseModel 看起来很完整（tenant/workspace 隔离、跨工作空间挂载、Per-KB RAG 配置覆盖），容易让人以为"知识治理已经做了"。但这些是访问控制，不是知识治理。真正的治理要求：知识有版本、有不可变性、与部署闭包绑定、变更触发新部署。当前代码连版本号字段都没有。

### 今天最大的决策
Knowledge Gap 的优先级排第一（高于 RuntimeABI 和 CapabilityRelease），因为：部署闭包不完整 = 执行不可复现 = "上周三数字员工回答了什么"无法追溯。在商业地产场景，这等于合同审批引用的政策版本不可追溯。RAG 工程能力（检索质量优化）可以后续迭代，但知识版本化是不可妥协的治理底线。

---


## 2026-08-14（Week11-Day5：竞品对比 — Dify/LangGraph/OpenClaw/Claude Code）

### 今天最大的认知
以前以为 LangChat 和 Dify 是同一赛道的竞品，区别只是"企业级 vs 开源社区"。现在知道它们根本不在同一条赛道上。Dify 解决的是"AI 应用怎么快速搭建和运行"——核心价值是速度。LangChat 解决的是"AI 应用怎么在企业治理框架下确定性运行"——核心价值是确定性。Dify 说"配置即应用，改了就生效"。LangChat 说"制品即应用，改了要走完整链路"。这两句话背后是完全不同的架构哲学、完全不同的目标客户、完全不同的商业模型。这不是"更好的 Dify"，而是"Dify 范式的反面"。

### 今天最大的坑
竞品对比最大的陷阱是"功能列表比较"——列一张表，打勾打叉，最后得出"我们功能更多"的结论。这毫无意义。真正的对比维度是架构范式：制品链 vs 配置即应用、不可变闭包 vs 可变运行时、前置治理 vs 后置治理。LangGraph 不是竞品——它是可以被 LangChat 封装在 SkillRelease 里的编排能力。OpenClaw 不是竞品——它是调用 LangChat 的 Agent Host。Claude Code 不在企业 AI 应用平台赛道。把不是竞品的东西当竞品，会浪费精力防守不存在的战线。

### 今天最大的决策
LangChat 最独特的设计可以用一句话概括：把软件工程的最佳实践（制品链、确定性构建、不可变部署）引入 AI 应用治理。这不是功能创新，是范式创新。这决定了 LangChat 的竞品不是 Dify/LangGraph，而是"企业用传统方式自建 AI 应用"这个现状。真正的竞争不是功能对比表，是"有没有制品链治理"——这是企业 CIO 能理解的语言。

---


## 2026-08-15（Week11-Day6：⚡ 实战交付 — LangChat v2 实施路线图 v1.0）

### 今天最大的认知
以前以为实施路线图就是把 Gap Matrix 里的红色项按优先级排序、估工作量。现在知道路线图的第一行不是任何 Gap，而是术语清理——"SkillRelease" 在 v1/v2 指不同的东西，这种术语重叠陷阱会让所有后续工作沟通失真，两个工程师讨论的是不同的东西还以为在讨论同一个。其次，排序依据是"运行时爆炸概率×合规影响"，不是"代码量大小"——3710 行的 v1 canonical execution 是最大代码资产但不是最危险处，最危险的是 DeploymentRevision"以为闭包完整实际不完整"的虚假安全感。

### 今天最大的坑
差点把 Connector（Day 3 结论的最薄弱环节）排进前 3 个 Sprint。但仔细想：tools-call-external-provider-guard spec 已经挡住了不安全的外部调用路径，段 3 空白是"能力缺失"不是"运行时爆炸"；而 Outbound System Bridge 的 Phase-0 Gate 只关 2/10、设计还在 review-blocked，抢跑等于在没有地基的地方盖楼。**"最薄弱"和"最紧急"是两个维度**——路线图排的是紧急度+依赖链，不是薄弱度排名。

### 今天最大的决策
前 3 个 Sprint 排序定为：Sprint 0（1周）术语清理+基线冻结 → Sprint 1（2周）KnowledgeSnapshot 补全部署闭包 → Sprint 2（2周）RuntimeABI+OCI 打通 → Sprint 3（2周）CapabilityRelease+发布流；Connector 留 Sprint 4+ 并行推 OSB Phase-0 Gate 证据（8/22 deadline）。7 周后达到「最小可治理制品链」。元规则：验收标准必须用代码事实定义，不用文档说法定义——否则路线图变成又一份自我安慰文档（Target Domain Model §1.2 的警告同样适用于路线图自己）。


## 2026-08-16（Week11-Day7：最终 Virtual CTO Review — 4 周总复盘）

### 今天最大的认知
以前以为架构评审的评分下降是坏信号。现在知道 W8-W11 五维评分从 7.2 下探到 6.4，与理解深度从 7.0 升到 9.0 形成的"剪刀差"，恰恰是学习有效的证明——W8 是拿望远镜打分（看到框架完整），W11 是拿内窥镜打分（看到三套平行体系、术语陷阱、闭包空洞）。同一架系统，测得越准，分越实。就像 ERP 上线前的数据体检：第一轮盘点说库存准确率 95%，逐仓细盘后变成 88%——库存没变，盘点精度变了。Day 6 路线图 v1.0 的产出意味着测量阶段结束、行动阶段开始：从今天起，五维评分的下行压力应当被 Sprint 交付逐步对冲。

### 今天最大的坑
Review 建议积压。复盘四周的 ADR Health Check：W9 建议 6 条（v2-ADR 冻结、007 拆分、implementation_status 字段等），W10 建议 PII 默认开启（一行代码的 P0 修复），到今天代码复核 pii_redaction.py:172 默认值仍是 False——**建议必须有 owner 和 deadline，否则架构评审退化成"合规表演"**。四周边写边发现 Gap，但 Review 只产出了认知，没有产出变更。这是导师模式的结构性盲区：它能看，不能改。

### 今天最大的决策
宣布"架构导师模式"收官，切换"开发搭档模式"。学习线继续（W12-13 Vision Intelligence），开发线启动 Sprint 0（术语清理 + PII 默认值修复一起提交，让建议落地率从 0/6 变成 2/6）。同时把五维评分改造为工程仪表盘：Sprint 1 验收绑定 Code Health 回升至 6.5+，Sprint 2 绑定 Technical Debt 回升至 6.0+，Sprint 3 绑定 ADR Consistency 回升至 7.0+——评分不再只是认知记录。留给 Week 15 的判据：如果那时评分还在下行，说明路线图没有被执行，那才是真正的问题。


## 2026-08-17（Week12-Day1 · Vision Intelligence 全景）

### 今天最大的认知
MallSenseAI 不是 CV 项目：CV 只占价值链前 1/5（Detection），核心资产是检测之后的业务闭环（规则→告警→工单→通知）。代码验证证实 PRD 判断——当前是封闭系统，无任何 capability 暴露，无法被上层编排。

### 今天最大的坑
发现 MallSenseAI 对外已更名为 LangChat AI Vision（ADR-004），但仓库代码模块名未动（保护契约）。读文档和读代码会看到两个名字，必须知道这是同一个东西，且重命名只覆盖对外品牌。

### 今天最大的决策
Capability 粒度倾向：业务级（safety.alert.query/subscribe）为主 + 少量原子级（vision.detect）。待周六画五层图时结合制造业场景验证配置类能力（vision.rule.configure）是否必要。


## 2026-08-18（Week12-Day2 · MallSenseAI 仓库精读：截图 vs 视频流）

### 今天最大的认知
采样方式是业务时间尺度的函数，不是技术能力的函数。状态型场景（消防通道占用/地面脏污/堆物）物理变化以分钟计，截图采样就是终局不是妥协；事件型场景（跌倒/入侵）瞬间不可重现，必须视频流。判断标准是"业务对象变化多快"，不是"有没有 GPU"。CameraAdapter 接口本身就体现了截图世界观（capture_snapshot() -> bytes 单张 JPEG），且 adapter 抽象为未来 RTSP 留了无侵入的口子。

### 今天最大的坑
规则引擎自称 Stateless evaluator，但"停留超时"规则靠 cooldown_state 里的 active_since 跨快照累计停留时长——第一眼以为是名不副实，细看才懂这是"无状态核心 + 状态外置"模式：引擎本身不持有状态，状态放外部 store。这和 LangChat 无状态 Runtime + 外部状态是同一个架构模式，两个产品在两个领域独立收敛到同一答案。

### 今天最大的决策
用"老人跌倒 10 秒告警"需求做了三层压力测试（采样层 600 倍采样率、检测层无状态逐帧接口失效、规则层缺 Tracking 跨帧去重），确认事件型场景需要的是架构升级不是参数调整。倾向把"采样策略 per-camera 配置化"记入周六五层图的设计约束——混合模式（关键路视频流 + 其余截图）才是商业地产现实。


## 2026-08-19（Week12-Day3 · Detection 体系：为什么选 YOLO-World）

### 今天最大的认知
模型选型是约束求解，不是排行榜浏览。MallSenseAI 的 models/ 目录躺着三份权重 + 一个无模型检测器，四个检测器四种策略（COCO 闭集 / D-Fire 微调 / YOLO-World 零样本 / 基线对比）——检测体系是谱系不是单品，每个检测器对应一档"词表开放度 × 精度责任 × 算力预算 × 数据成本"的权衡。YOLO-World 入选的唯一性由三条硬约束推出：商场障碍物长尾词表不在 COCO 80 类（15 个默认类里 9 个 COCO 没有）→ 闭集出局；CPU-only 单帧预算 → GroundingDINO 出局；"暂不推广但维护"零标注预算 → 微调只配给火灾这种法律责任级场景。它的精度短板（min_confidence 只敢设 0.25）用管线四层兜底补：低阈值召回 → ROI 质心+面积比双闸门 → 规则引擎 → Cooldown。误报率是系统属性，不是模型属性。

### 今天最大的坑
细读 yolo_world.py 发现 classes 不是代码常量而是从 detector_configs 表读的运行时参数，ConfigWatcher 每 10 秒轮询热更新、多副本原子快照切换——第一反应是"这么重？"。细想才懂这是产品级必要设计：零样本检测的全部价值就在"检测什么"可运营配置化，如果改个词表要重启，开放词表就退化成了"换个地方硬编码"。顺带捡到一个产品级 bug 修复的教科书案例：文件头 monkey-patch ultralytics CLIP.encode_text 修 GPU 设备错位，AGENTS.md 第 127 条专门警告不要简化回去——长期运营资产和 demo 代码的区别就在这些注释里。

### 今天最大的决策
回答架构师思考题 1（共享充电宝检测）：先加文本类零样本试运行，用误报率数据驱动"何时转微调"——alarm_images 的 21 个目录 38MB 告警快照天然就是微调训练集的候选池，误报样本库 = 未来数据闭环的原材料。这个"零样本先上、数据回流、够痛再微调"的演进路径记入周六五层图备注，和 LangChat 侧"先 Capability 后 Governance"的节奏是同一个元策略：先让能力跑起来，用真实运行数据决定加码哪里。


## 2026-08-20（Week12-Day4 · Video Analytics 基线：从截图到视频流要改什么）

### 今天最大的认知
截图→视频流不是换采集协议，是执行范式转换：拉动模型（scheduler 到点才算 due、请求响应、处理释放）→ 推送模型（帧持续到达、消费者常驻、状态常驻）。五层耦合联动：采集（无状态 HTTP→RTSP 长连接会话）、处理（定时拉→帧队列推）、推理（0.35 次/秒→525 帧秒，1500 倍，CPU→GPU 必选）、状态（cooldown 外置→Track 常驻内存）、证据（JPEG→环形缓冲剪辑）。而 BaseDetector.detect(image_bytes) 契约一行不用改——改造全在检测器上游和下游，Day2 看到的 adapter 隔离在这里真正值钱。硬信号：config 里 alarm_interval_minutes=1（分钟）与 fire_smoke_check_interval_seconds=15（秒）并存，采样率压力已在参数层现形。

### 今天最大的坑
差点把"全流 25fps"当成视频化的默认答案。算完账才发现被忽略的中间态 B（RTSP 抽帧 1-2fps）才是架构上自然的下一站：保留检测契约、只重写采集层、21 次推理/秒 GPU 轻松扛住、且正好接住火灾 15 秒采样的业务压力。还有一层：OpenSpec 29 个 spec 里没有任何 video/stream/tracking 字样——视频流连规格层都还没进，是 roadmap 谈话不是 reality；而 AGENTS.md 里"password 存明文因为 HTTP/RTSP 都要用"和 legacy 里 update_base_image.py 的 4 种 RTSP URL 尝试，说明 RTSP 凭据和代码足迹早就预留，只是主链路从未启用——数据模型留了门，规格层没立户。

### 今天最大的决策
确立了演进谱系决策框架 A→B→C→D（截图加密→RTSP 抽帧→全流+GPU+Tracking→边缘盒子），每档按"改造量/帧率/事件覆盖/单路成本"定价。判断变量是场景 ROI 不是技术成熟度：跌倒检测防人身诉讼、客流统计支撑租金定价才配得上 C 档 10 万元级硬件；消防通道堆物留在 A 档零成本。竞争位锁定在 L3-L4（告警→工单→通知业务闭环）而非 L1-L2（海康大华盒子地盘）。明天 Day5 的 Business Scene Matrix 就用这条谱系当标尺，给每个商业场景标"需要站到哪一级"。


## 2026-08-21（Week12-Day5 · Business Scene Matrix：哪个商业场景 ROI 最高）

### 今天最大的认知
ROI 排序不按技术先进度也不按价值金额排，按"预算科目 × 资产复用度 × 信任成本"排。三种价值货币（风险规避=保费逻辑/人力替代=工资逻辑/收入增量=租金逻辑）落在两个预算科目（费用/资本）里——消防通道占用行边际成本≈0（21 路摄像头已沉没、5 个规则模板已跑通、新摄像头自动绑定默认规则）价值双算（合规风险+替代 252 次/天巡检），是当期 ROI 王；客流统计金额百万级但属资本预算且要 C 档硬件，是战略行不是当期行。误报率是经济学指标不是技术指标：它通过告警疲劳把三类价值同时翻负，而四层兜底+参数自主可调把它变成运营杠杆。最被低估的是 L5 自动巡检日报——复用已有告警数据资产，边际成本最低的收入型场景，Vision Agent 的最小可行形态。

### 今天最大的坑
差点把 Business Scene Matrix 当需求分析文档写。写完才发现它同时是三样东西：Capability 需求清单（哪行值得按 ADR-003 暴露）、预算答辩武器（费用 vs 资本科目）、路线图输入（档位列映射 Day4 的 A→D 谱系）。另外撞上产品定位与能力地图的张力：域知识.md 明确"不做客流统计"，但客流是五层模型 L3-L4 核心场景——这不是矛盾，是"商业定位（巡检告警产品）≠ 能力地图（Vision Intelligence 全谱）"，矩阵里必须区分 reality 行和 roadmap 行（OpenSpec 29 个 spec 无 video/stream/tracking 是硬判据）。

### 今天最大的决策
矩阵判定结论记入周六画图输入：①当期主打行=消防通道（已有资产复用叙事）；②战略行=客流（资本预算科目，二期答辩用）；③增量行=违停/占道/垃圾满溢（A 档零样本改词表即扩场景，把扩场景从硬件采购降级为配置变更——这是矩阵里最重要的架构经济学事实）；④优先验证行=L5 日报（等 Day6 评估读 DB 直连 vs 走 Capability 两条路线的成本）。架构师思考题 2 的结论倾向：火灾检测合同措辞必须卖"降低概率"不卖"消除风险"，漏报率不量化就不进第一期 SLA 承诺。


## 2026-08-22（W12-D6 周六交付：五层图 + Business Scene Matrix）

### 今天最大的认知
五层模型是 DAG（能力依赖的分类账），不是楼梯（升级路线）。代码级证据：workers/pipeline.py 五站链路（capture→detect→persist→rule→alert）中第 4 站（规则统计）不经过任何 L2 组件——状态型场景从 L1 直接记账到 L4'（工单闭环），跳 L2 是合法架构决策不是缺陷。据此修正了对 MallSenseAI 的定位：不是"低级 L1 系统"，是"窄而深"——L1 六格占四（含 P2 半格）、L3 退化统计两个（duration/area 是无跟踪的 Scene Understanding）、L4 闭环雏形、L2 有意为零（域知识"不做什么"三条 = 主动边界）。初级和专注在图上长得一样，区别在边界是画的还是没爬到的。

### 今天最大的坑
ipynb 实验 3 的行人计数模拟掉进自己挖的坑两次：① 一维世界（所有人同一条线）导致跟踪器身份根本歧义，贪心最近邻偷换轨迹、吞人计数；② 去掉距离闸门后新入场者挂到远处旧轨迹上，把已计数的 tid 复用造成假漏计。修复：2D 泳道世界 + 仅匹配上一帧更新过的轨迹 + 0.04 距离闸门，得到 43/40（~7% ID 切换误差——本身就是真实跟踪系统的教学点）。这个坑恰好演示了 md §9 的论断：L3 退化形态的边界来自 L2 缺失，无 ID 时慢行者被系统性高估（平均停留 80 帧 = 80× 过计数），静态误报（广告画假人）在 L1 下是随帧率线性放大的永久偏差。

### 今天最大的决策
① "MallSenseAI 在 L1 哪里"的正式答案定为格子间坐标而非单点：四格（封闭词表 yolo11n / 领域微调 D-Fire / 开放词表 YOLO-World / 基线比对 absdiff）+ 运行时路由（service.py:63 "prefer YOLO-World, fallback to Debris"——L1 内部已有容错拓扑）。② Business Scene Matrix 升级为层覆盖版（9 行 × 所需层/已有层/层缺口），三条硬结论进交付物：当期 ROI 行 = 层缺口 0 行；L2 是全部 B/C 档行的单点依赖层；L5 日报反而地基最全（L1+L3'+L4'），是最便宜的升层路径——Day5 判定④有了架构学依据。③ 技术债三件记入明日评分：DetectorType 枚举无 yolo_world/floor_cleanliness 位、AGENTS.md detectors 目录注释缺两个文件、Camera.password_hash 列名存明文。


## 2026-08-23（W12-D7 周日 · Virtual CTO：MallSenseAI 能力边界审视 + 五维评分）

### 今天最大的认知
横向对照发现平台本体（LangChat，W11 Code Health 6.0）的代码健康度低于挂在它下面的行业应用（MallSenseAI，7.5：510 后端测试 + 37 e2e + CI 三段流水线）。架构先进性和工程纪律是两个独立计分项——ADR 华丽救不了测试空白，测试纪律也不需要先进架构护航。集成时的信任边界要按 Code Health 画，不按 ADR 画。另一个定稿认知：边界判定三准则（状态型 vs 事件型 / 闭环可审计 / 人力替代 ROI）可以机械化地跑任何新场景，跑 L2 视频流的结论是"值得做但必须作为独立数据契约的新管线"——验证并定稿了 D4 结论。

### 今天最大的坑
ADR-004（MallSenseAI → LangChat AI Vision）声明的三项能力——客流/商品结构化分析、零售 POS/CRM 事件联动、通过 LangChat Channel 回传洞察——代码为零，29 个 OpenSpec spec 无任何支撑。加上 ADR-008（8/21，本周五）langchat→lnkchat 改名落地，MallSenseAI 对外名两周内换了两次（MallSenseAI→LangChat AI Vision→LnkChat AI Vision）。"行业能力包"目前只存在于 ADR 文本：两边各自完整（平台有治理体系、应用有闭环+测试），中间的桥（Capability 注册/Channel 集成）完全没画。

### 今天最大的决策
① MallSenseAI 首评 7.05（AQ 7.0 / CH 7.5 / ADR 6.0 / TD 7.0 / DX 7.5），明确记为"望远镜分数"并预测内窥镜区间 6.3-6.8，W13-D7 复评验证——把 W11 剪刀差方法论从 LangChat 移植到新对象。② 战略结论定稿：未来两季度主战场 = L1 换格子（YOLO-World 改词表扩场景）+ L4 补全（工单闭环指标化），不爬 L2；L2 等场景饱和 + 算力预算到位再作为独立管线启动。③ 术语切换三策略自 W13 生效：历史文档不追改、新笔记用 LnkChat（首现标注）、仓库/库/端口不动（ADR-004+008 双确认）。④ ADR-004 建议在 §1 三项纯规划承诺处标注"目标态"，防止销售/实施误读为现状态。


## 2026-08-25（W13-D2 Security Analytics：误报率为什么是核心挑战）

### 今天最大的认知
误报率在安全场景不是质量指标，是商业模式否决项。base rate 接近零（商场一年真火警 0~1 次）时，模型精度和有效告警质量是两个指标：99% 单帧精度对着每天数百次截图评估，年产误报仍是真报的几十倍——先验概率淹没精度，换更强模型（RT-DETR/大 backbone）救不了。真正的杠杆在模型之外的三层漏斗，且代码里已全部存在：①检测器层四道闸门（conf/label 白名单/area_ratio/ROI 中心点过滤，fire_smoke.py）；②规则引擎时间维度确认（min_stay_seconds 把"瞬时误检"和"持续事件"分开 + cooldown 治重复告警，engine.py 负键 first-seen/正键 last-alert）；③人审社会裁决（AlertStatus.false_positive 一等公民状态，pending→confirmed/false_positive/resolved）。六场景误报治理难度排序恰好等于 Tracking 依赖度排序——误报率是 L1→L2 升级的真实架构动机。

### 今天最大的坑
差点把"误报"当成单一概念处理。拆开才发现至少三种：瞬时误检（单帧噪声，min_stay 治）、持续误报（真有东西但不是违规，如过路搬运，duration/threshold 治）、重复告警（同一事件反复推，cooldown 治）。三种的治理手段完全不同，混在一起调阈值必然顾此失彼。另一个发现级细节：FireSmokeDetector 每个 detection 的 metadata 都带 supplemental: True——视频烟火是烟感的补充不是替代，"不做唯一真相源"本身就是误报治理（法律责任分摊），代码化石里藏着架构决策。

### 今天最大的决策
判定当前最大的 Gap 不是漏斗缺失而是闭环缺失：false_positive 人审标记被结构化留存（还有 Redis HSETNX 保证多 worker 竞争一致），但不回流——不调阈值、不进困难样本集、不驱动任何学习回路。人审在做分类，系统不在学习。修复优先级排序（记入周六 Radar 输入）：短期=阈值回流（用误报标记统计自动建议 confidence_threshold 调整，反馈周期最短）；中期=ROI×时段维度误报画像（蒸汽/夕照类误报有规律性时空分布）；长期=困难样本微调（数据量门槛高，最后做）。同时确认边界设计即治理：域知识"不做消防联动"把误报爆炸半径限制在"人跑一趟"，SLA 只能卖"降低概率"不能卖"消除风险"。


## 2026-08-26（W13-D3 Retail Analytics：视觉能力如何转化为 KPI）

### 今天最大的认知
零售场景的真正瓶颈不在检测在度量衡：像素→检测框→事件→指标→KPI→决策这条装配线上，检测框之后每一步都是测量问题。三个定量结论（notebook 仿真验证）：① 排队等待 W 按 Day1 谱系需要 ID 才能"测"，但 Little's Law `W = L/λ` 用两个无 ID 可测量（队长、POS 到达率）就能"算"——而且病态区反直觉地不在高峰在低峰（分母 λ̂ 趋零，除法放大误差，30 天仿真低流量窗口误差 std 高 1.4×），KPI 报表窗口必须按流量分级；② 检测噪声 vs 检测偏差是两种病：噪声按 1/√N 自动缩减（25 天仿真验证 √N×std 恒定），偏差是永远不掉的地板（−15% 遮挡 → 恒 −0.4 分），加密抽样治不了偏差，高峰段 2 小时人工清点一次校准就能拆掉地板（残差 −0.06 分）——钱优先花在校准；③ OSA 类比例 KPI 的检测混淆（8% 漏检+2% 误报）直接写进月度数字：45,000 次检查平均后抽样噪声塌缩到 ±0.0013，但 −1.3% 偏差纹丝不动，"不校准的报表越平滑越稳定地错"。

### 今天最大的坑
两次被自己的直觉骗，都被 notebook 数据打脸：先验认为"午高峰非稳态 → Little 估计偏差大"，实测恰恰相反（高峰样本密集最贴线，低峰除法病态）；第一版检测模型用 round+clip 模拟计数，结果在空场景引入 +0.5 分虚假系统偏差（clip 的 Jensen 效应），把噪声/偏差实验污染成三向纠缠，换成乘法零均值噪声模型才干净。教训记两条：KPI 类结论必须先跑仿真再写结论；"检测误差对称"是物理上不成立的理想化（真实检测错误以漏检为主，天然负偏）。

### 今天最大的决策
① 判定 Retail Analytics 是 MallSenseAI 打开封闭系统的首选切口：域知识设计决策 #2 封闭的理由是"安防秒级实时性不容编排中转"，反读即"分钟级容忍场景无理由封闭"——零售 KPI 恰好分钟/日级容忍，`retail.kpi.query` 应排在 `safety.alert.query` 之前成为第一个暴露的 capability（用旧 ADR 的边界条件论证新方向，零新增证据）。② 硬 Gap 定位：全仓库无 Metric/TimeSeries 领域对象（entities.py 全是事件型 Alert 状态机），"平均等待"在这套领域模型里无处安放——需要 MetricSample/KPIDefinition/CalibrationRecord 新对象，ADR 级决策不是加张表；检测层反而 60% 现成（yolo_world 换 prompt 即 person/empty shelf，OpenSpec 已有词表热配置场景）。③ 场景级质量策略应进 capability 元数据：安全误报=信任死刑（多层漏斗压制）、零售偏差=慢性毒（一次性校准+大数定律），同一套检测基础设施配不同质量工程——Day2 的"错误经济学分场景"升级为显式建模要求。


## 2026-08-27（W13-D4）Vision Agent vs LangChat Agent

### 今天最大的认知
"Agent"的自治有三种时钟——意图驱动（用户请求）、时间驱动（班次/跑批）、事件驱动（告警），时钟不改变物种。当前 MallSenseAI pipeline 是 automation 不是 agent（规则触发 vs 证据推理的分界）；Vision Agent 不是第三个物种，= L1-L4 感知系统 + LnkChat 数字员工的组合体。真正的结构差异在三件事：**时钟（谁触发）、证据（凭什么推理——LangChat Agent 推理结构化事实，Vision Agent 推理统计量，必须携带置信度说话）、失败模式（fail-closed 拒绝 vs 永不停机降级）**。

### 今天最大的坑
容易把 Vision Agent 想成"CV 领域的另一套 agent 框架"（perception-action loop 自治体），然后陷入选型思维。实际上框架问题平台层已经解决（SkillRelease + DigitalEmployee，ADR-LC-013 的 dispatch guard 代码已落地），剩下的是证据学问题：置信度传播、口径责任、日报公信力。域知识.md 的封闭系统边界在日报场景（分钟/日级容忍）再次失效——和 D3 一样的论证路径。

### 今天最大的决策
L5 推理层选址三案裁决：A 封闭系统内自建 LLM 调用（治理全缺）❌ / B 独立 Vision Agent 产品（双份 runtime 双份治理）❌ / **C 平台 Skill + 视觉 Capability**（MallSenseAI 到 L4 为止，推理住 LnkChat Skill 层）✅。核心理由：变化率分层——检测模型按周变、prompt 按天变，capability 接口把两种变化率切开。前置件：capability 注册表（P0）、日级调度触发源、证据 digest 对齐。

### 遗留 / 下一步
- D5（明天）：MallSenseAI 进入 LnkChat 的身份裁决（Connector / Capability 提供方 / Agent Host，今天思考题①即开场题）+ 集成路径
- 四个不存在的 gap 待 D5 排序：日级聚合层、指标口径层、LLM 推理层、capability 出口
- ADR Health Check 候选：域知识.md「不做客流统计」边界声明（D3 已登记，集成时必须重新裁决）
